In [64]:
import os
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

In [65]:
from dotenv import load_dotenv
load_dotenv()

True

In [66]:
class AgentState(TypedDict):
    # Input
    query: str
    user_id: str
    grade: str                          # e.g. "9th"
 
    # Routing decisions
    route: Literal["class_agent", "personalized"]
    query_type: Literal["numerical", "concept", "both"]
 
    # Retrieved data
    rag_context: Optional[str]
    personalized_data: Optional[dict]
 
    # Final response
    response: str
    needs_more_context: bool     

In [67]:
from langchain.chat_models import init_chat_model

# llm = ChatGroq(model="llama-3.1-8b-instant")
llm = init_chat_model("groq:llama-3.1-8b-instant")

In [68]:
def get_rag_context(query: str, grade: str) -> tuple[str, bool]:
    """
    Stub: Replace with your actual vector store retrieval.
    Returns (context_string, is_sufficient).
    """
    # Example: context = vectorstore.similarity_search(query, k=5)
    context = f"[RAG] Grade {grade} context for: {query}"
    is_sufficient = len(context) > 20            # replace with real scoring
    return context, is_sufficient

In [69]:
def get_personalized_data(user_id: str) -> dict:
    """
    Stub: Replace with your user DB / profile service call.
    """
    return {
        "user_id": user_id,
        "weak_topics": ["trigonometry", "quadratic equations"],
        "strong_topics": ["basic algebra"],
        "learning_style": "visual",
        "past_errors": ["sign errors in equations"],
    }

In [70]:
def routing_agent(state: AgentState) -> AgentState:
    """
    Decides whether the query goes to the class-based agent
    or directly to personalized-data delivery.
    """
    prompt = f"""You are a routing agent for an educational system.
Classify the query into ONE of these routes:
- "class_agent"   : academic / subject question (math, science, etc.)
- "personalized"  : request for student progress, recommendations, weak topics
 
Query: "{state['query']}"
Respond with ONLY one word: class_agent OR personalized"""
 
    result = llm.invoke([HumanMessage(content=prompt)])
    route = result.content.strip().lower()
    if route not in ("class_agent", "personalized"):
        route = "class_agent"           # safe default
 
    return {**state, "route": route}

In [71]:
def class_based_agent(state: AgentState) -> AgentState:
    """
    Handles academic queries for grade 9.
    Classifies into numerical / concept / both.
    Also fetches required RAG context.
    """
    # Classify query type
    type_prompt = f"""Classify this 9th-grade academic query:
- "numerical"  : requires calculation / problem solving
- "concept"    : requires conceptual explanation
- "both"       : requires both
 
Query: "{state['query']}"
Respond with ONLY one word."""
 
    type_result = llm.invoke([HumanMessage(content=type_prompt)])
    query_type = type_result.content.strip().lower()
    if query_type not in ("numerical", "concept", "both"):
        query_type = "concept"
 
    # Fetch RAG context
    rag_context, is_sufficient = get_rag_context(state["query"], state["grade"])
 
    return {
        **state,
        "query_type": query_type,
        "rag_context": rag_context,
        "needs_more_context": not is_sufficient,
    }

In [72]:
def numerical_solver(state: AgentState) -> AgentState:
    """Solves numerical / calculation problems using RAG context."""
    system = f"""You are a 9th-grade math/science tutor.
Use ONLY the provided context. Show step-by-step working.
 
Context: {state.get('rag_context', '')}"""
 
    result = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=state["query"]),
    ])
    return {**state, "response": result.content}

In [73]:
def concept_understander(state: AgentState) -> AgentState:
    """Explains concepts using RAG context."""
    system = f"""You are a 9th-grade conceptual tutor.
Use ONLY the provided context. Give clear, simple explanations with examples.
 
Context: {state.get('rag_context', '')}"""
 
    result = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=state["query"]),
    ])
    return {**state, "response": result.content}

Peronalized Data

In [ ]:
def personalized_data_node(state: AgentState) -> AgentState:
    """
    POST-PROCESSOR — runs after every solver node.
 
    Takes the raw academic response and wraps it with:
      - A note on how this topic relates to the student's known weak areas
      - A tip tailored to their learning style
      - An encouragement hook tied to their strongest subject
      - A "watch out for" flag if this topic overlaps with past errors
 
    The raw answer is NOT changed — only context is appended.
    """
    profile = get_personalized_data(state["user_id"])
    raw_response = state.get("response", "")
 
    system = """You are a personalized learning coach.
You receive a complete academic answer and a student's profile.
Your job is to ADD a short "Personalized Note" section at the END of the answer.
 
The note must:
1. Connect this topic to the student's known weak areas (if relevant).
2. Give ONE study tip matched to their learning style.
3. Flag any past error patterns that could trip them up here.
4. Be encouraging but specific — no generic praise.
 
Do NOT rewrite or summarize the academic answer. Only append the note."""
 
    user_msg = f"""Academic answer:
{raw_response}
 
---
Student profile:
- Weak topics     : {profile['weak_topics']}
- Strong topics   : {profile['strong_topics']}
- Learning style  : {profile['learning_style']}
- Past errors     : {profile['past_errors']}
- Grade           : {state['grade']}
 
Append a " Personalized Note" section to the answer above."""
 
    result = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=user_msg),
    ])
    return {**state, "personalized_data": profile, "response": result.content}

In [75]:
def rag_node(state: AgentState) -> AgentState:
    """
    Central RAG retrieval gate.
    Called when context is sufficient — passes enriched context downstream.
    """
    # Context was already fetched in class_based_agent; this node
    # can optionally re-rank or expand it.
    enhanced_context, _ = get_rag_context(state["query"], state["grade"])
    return {**state, "rag_context": enhanced_context}

In [76]:
def route_after_routing_agent(state: AgentState) -> str:
    return state["route"]                       # "class_agent" | "personalized"
 
 
def route_after_class_agent(state: AgentState) -> str:
    qt = state.get("query_type", "concept")
    if qt == "numerical":
        return "numerical"
    elif qt == "concept":
        return "concept"
    else:
        return "both"                           # handled by concept node with calc
 

In [77]:
def build_graph() -> StateGraph:
    graph = StateGraph(AgentState)
 
    # Register nodes
    graph.add_node("routing_agent",        routing_agent)
    graph.add_node("class_based_agent",    class_based_agent)
    graph.add_node("numerical_solver",     numerical_solver)
    graph.add_node("concept_understander", concept_understander)
    graph.add_node("personalized_data",    personalized_data_node)
    graph.add_node("rag",                  rag_node)
 
    # Entry point
    graph.set_entry_point("routing_agent")
 
    # Routing agent → class_based_agent (single route now)
    graph.add_edge("routing_agent", "class_based_agent")
 
    # Class agent → numerical | concept
    graph.add_conditional_edges(
        "class_based_agent",
        route_after_class_agent,
        {
            "numerical": "numerical_solver",
            "concept":   "concept_understander",
            "both":      "concept_understander",
        },
    )
 
    # RAG enriches context before solvers (optional pre-retrieval step)
    graph.add_edge("rag", "concept_understander")
 
    # Both solvers → personalized_data (always — it's a post-processor)
    graph.add_edge("numerical_solver",     "personalized_data")
    graph.add_edge("concept_understander", "personalized_data")
 
    # Personalized data → END
    graph.add_edge("personalized_data", END)
 
    return graph

In [78]:
memory   = MemorySaver()
workflow = build_graph()
app      = workflow.compile(checkpointer=memory)

In [79]:
def run_query(
    query: str,
    user_id: str = "student_001",
    grade: str   = "9th",
    thread_id: str = "default",
) -> dict:
    """
    Main entry point.
 
    Args:
        query     : Student's question.
        user_id   : Unique student identifier.
        grade     : Student's grade (default "9th").
        thread_id : LangGraph thread for conversation memory.
 
    Returns:
        Full state dict including 'response'.
    """
    initial_state: AgentState = {
        "query":             query,
        "user_id":           user_id,
        "grade":             grade,
        "route":             "class_agent",      # overwritten by routing_agent
        "query_type":        "concept",          # overwritten by class_based_agent
        "rag_context":       None,
        "personalized_data": None,
        "response":          "",
        "needs_more_context": False,
    }
 
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(initial_state, config=config)
    return result
 

In [80]:
if __name__ == "__main__":
    tests = [
        "Solve: 2x² + 5x - 3 = 0",
        "Explain Newton's second law of motion",
        "What are my weak topics and how can I improve?",
    ]
    for q in tests:
        print(f"\n{'='*60}\nQuery: {q}")
        out = run_query(q, user_id="student_42", thread_id="test")
        print(f"Route      : {out['route']}")
        print(f"Query type : {out.get('query_type', 'N/A')}")
        print(f"Response   : {out['response'][:300]}...")


Query: Solve: 2x² + 5x - 3 = 0
Route      : class_agent
Query type : numerical
Response   : Academic answer:
To solve the quadratic equation 2x² + 5x - 3 = 0, we can use the quadratic formula:

x = (-b ± √(b² - 4ac)) / 2a

In this equation, a = 2, b = 5, and c = -3.

Step 1: Plug in the values of a, b, and c into the quadratic formula.
a = 2, b = 5, c = -3

Step 2: Calculate the value of b...

Query: Explain Newton's second law of motion
Route      : class_agent
Query type : concept
Response   : Academic answer:
Let's break down Newton's second law of motion in simple terms.

**What is Newton's second law of motion?**

Newton's second law of motion states that the force applied to an object is equal to its mass multiplied by its acceleration. This is often written as:

F = ma

Where:
- F is...

Query: What are my weak topics and how can I improve?
Route      : personalized
Query type : concept
Response   : Academic answer:
To figure out your weak topics and improve them, we'll do a si